In [46]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot
import numpy as np
import os
from datetime import timedelta

pd.set_option('display.max_columns',None)

In [47]:
def get_dataset(dataset:str)->str:
    try:
        return pd.read_csv(os.path.join('..','..','data','raw',f'{dataset}.csv'))
    except Exception as e:
        warnings.warn(f'"{dataset}" dataset not present.')
        return None

In [48]:
def calculate_distance(record):
    lat1 = np.radians( record['customer_latitude'])
    lng1 = np.radians(record['customer_longitude'])
    lat2 = np.radians(record['seller_latitude'])
    lng2 = np.radians(record['seller_longitude'])

    # Calucate the difference.
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    
    a = (np.sin(dlat/2)**2) + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2

    c =  2 * np.asin(np.sqrt(a))

    r = 6371

    return np.round((c * r),0)
    

In [49]:
data = pd.read_csv(os.path.join('..','..','data','processed','training_data.csv'))

In [50]:
data.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude
0,f15aaf3f7b6fbea7b5a454a4f9a38170,7a40fc1758e9b16f982192f2ce2736ea,delivered,2018-06-27 16:50:35,2018-06-27 17:51:04,2018-07-03 12:48:00,2018-07-10 20:22:12,2018-07-23,1.0,credit_card,2.0,155.95,1f79aef39bbaea74890551c00f8944c0,13256,itatiba,SP,-23.009681,-46.853418,1.0,0182db9fc95fd36e324e8a4c40c0819a,0c8380b62e38e8a1e6adbeba7eb9688c,2018-07-03 17:51:04,139.90,16.05,bebes,54.0,975.0,2.0,2400.0,35.0,10.0,30.0,baby,37410.0,tres coracoes,MG,-21.694107,-45.257984
1,da25492a616ea75359094ce9180d2691,cd5197b832fad889118cb4ec1483b4f2,delivered,2017-12-01 09:59:13,2017-12-01 11:31:06,2017-12-04 21:34:42,2017-12-07 17:14:48,2017-12-20,1.0,credit_card,2.0,188.80,5ed68fbea058394fe9017715f735f89e,9370,maua,SP,-23.677154,-46.463730,2.0,4c2394abfbac7ff59ec7a420918562fa,cc419e0650a3c5ba77189a1882b7556a,2017-12-08 10:13:07,84.99,9.41,beleza_saude,49.0,1495.0,3.0,600.0,33.0,19.0,13.0,health_beauty,9015.0,santo andre,SP,-23.658213,-46.523005
2,2c494cef88b908df152b7d6d69c8beba,f64b4d4b9e4185ce59b00e617e565bca,delivered,2018-05-10 15:49:11,2018-05-11 11:37:28,2018-05-18 14:02:00,2018-06-07 21:08:57,2018-05-28,1.0,credit_card,4.0,146.68,3af90f255e824d1d53b3584a4e89ad58,35162,ipatinga,MG,-19.469807,-42.563226,1.0,1405f399d2e4807af6ce6a2519e2ed7b,973f21788dfab357250f69a8dcb7ddee,2018-05-17 11:37:28,105.00,41.68,esporte_lazer,45.0,599.0,1.0,14550.0,20.0,25.0,13.0,sports_leisure,35530.0,claudio,MG,-20.441774,-44.766286


In [51]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 85068 entries, 0 to 85067
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       85068 non-null  str    
 1   customer_id                    85068 non-null  str    
 2   order_status                   85068 non-null  str    
 3   order_purchase_timestamp       85068 non-null  str    
 4   order_approved_at              84954 non-null  str    
 5   order_delivered_carrier_date   83601 non-null  str    
 6   order_delivered_customer_date  82657 non-null  str    
 7   order_estimated_delivery_date  85068 non-null  str    
 8   payment_sequential             85065 non-null  float64
 9   payment_type                   85065 non-null  str    
 10  payment_installments           85065 non-null  float64
 11  payment_value                  85065 non-null  float64
 12  customer_unique_id             85068 non-null  str    
 1

In [52]:
# Convert columns to their appropriate datatypes.

date_col = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date','shipping_limit_date']

number_col = ['payment_sequential','payment_installments','order_item_id','seller_zip_code_prefix']

for col in date_col:
    data[col] = pd.to_datetime( data[col])
    

for col in number_col:
    data[col] = data[col].astype('Int64')



In [53]:
data = data[data['order_status'] == 'delivered']

In [54]:
df_geolocation = get_dataset('geolocation')

In [55]:
df_unique_geolocation = df_geolocation.groupby('geolocation_zip_code_prefix').agg(latitude = ('geolocation_lat','median'),longitude=('geolocation_lng','median')).reset_index()

In [56]:
data[data['customer_latitude'].isna()]['customer_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [57]:
data[data['seller_latitude'].isna()]['seller_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [58]:
# Those zip code are not present in geolocation table so we need to drop them.

data.dropna(axis=0,subset=['customer_latitude','customer_longitude','seller_latitude','seller_longitude'],inplace=True)

In [59]:
data.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude
0,f15aaf3f7b6fbea7b5a454a4f9a38170,7a40fc1758e9b16f982192f2ce2736ea,delivered,2018-06-27 16:50:35,2018-06-27 17:51:04,2018-07-03 12:48:00,2018-07-10 20:22:12,2018-07-23,1,credit_card,2,155.95,1f79aef39bbaea74890551c00f8944c0,13256,itatiba,SP,-23.009681,-46.853418,1,0182db9fc95fd36e324e8a4c40c0819a,0c8380b62e38e8a1e6adbeba7eb9688c,2018-07-03 17:51:04,139.90,16.05,bebes,54.0,975.0,2.0,2400.0,35.0,10.0,30.0,baby,37410,tres coracoes,MG,-21.694107,-45.257984
1,da25492a616ea75359094ce9180d2691,cd5197b832fad889118cb4ec1483b4f2,delivered,2017-12-01 09:59:13,2017-12-01 11:31:06,2017-12-04 21:34:42,2017-12-07 17:14:48,2017-12-20,1,credit_card,2,188.80,5ed68fbea058394fe9017715f735f89e,9370,maua,SP,-23.677154,-46.463730,2,4c2394abfbac7ff59ec7a420918562fa,cc419e0650a3c5ba77189a1882b7556a,2017-12-08 10:13:07,84.99,9.41,beleza_saude,49.0,1495.0,3.0,600.0,33.0,19.0,13.0,health_beauty,9015,santo andre,SP,-23.658213,-46.523005
2,2c494cef88b908df152b7d6d69c8beba,f64b4d4b9e4185ce59b00e617e565bca,delivered,2018-05-10 15:49:11,2018-05-11 11:37:28,2018-05-18 14:02:00,2018-06-07 21:08:57,2018-05-28,1,credit_card,4,146.68,3af90f255e824d1d53b3584a4e89ad58,35162,ipatinga,MG,-19.469807,-42.563226,1,1405f399d2e4807af6ce6a2519e2ed7b,973f21788dfab357250f69a8dcb7ddee,2018-05-17 11:37:28,105.00,41.68,esporte_lazer,45.0,599.0,1.0,14550.0,20.0,25.0,13.0,sports_leisure,35530,claudio,MG,-20.441774,-44.766286
3,3a8d035c2ed963a9abdb531ed8a34086,a70d2aa8d067f4ec14c3cf080d6f820a,delivered,2017-07-15 10:57:57,2017-07-15 11:10:14,2017-07-19 22:12:55,2017-07-28 16:28:00,2017-08-28,1,credit_card,3,66.69,ef5a31b0437ed886c3113e8af13bdc06,48180,entre rios,BA,-11.943259,-38.085021,1,1ec486885049bbb9b79351d150ed18c4,cab85505710c7cb9b720bceb52b01cee,2017-07-20 11:10:14,49.90,16.79,fashion_bolsas_e_acessorios,36.0,412.0,5.0,100.0,16.0,5.0,20.0,fashion_bags_accessories,2252,sao paulo,SP,-23.478747,-46.588240
4,9bf784212164c8cc81d3eb16daaa5950,4acfe2f36c9f7b470ef68d71dc5b6ebc,delivered,2018-05-24 01:18:41,2018-05-24 01:38:03,2018-05-24 13:55:00,2018-06-06 19:04:51,2018-06-18,1,credit_card,1,203.68,2561161bc52cafa1be6e80ed9c92c9ea,17120,agudos,SP,-22.470734,-48.989212,1,ee4329a32508827791e17cd8daeadb6b,8648b1e89e9b349e32d3741b30ec737e,2018-05-28 01:31:24,189.00,14.68,construcao_ferramentas_construcao,55.0,721.0,2.0,650.0,16.0,11.0,11.0,construction_tools_construction,12308,jacarei,SP,-23.304380,-45.962662


In [60]:
# to order_approved at null values filling , first we take the mean time of order approved from order purchased which is '10 hourse' 
# then we fill the null values using order_purchase_time + 10 hours (which is aproximate)




duration = timedelta(hours=10)
data['order_approved_at'] = data['order_approved_at'].fillna(value= (data['order_purchase_timestamp']+duration))

In [61]:
# the remaining null values count are very low so we drop them directly before droping '82221 records'

data.dropna(inplace=True)


In [62]:
data['delivery_distance_km'] = data.apply(calculate_distance,axis=1)

In [63]:
# save data
data.to_csv(os.path.join('..','..','data','processed','cleaned_trained_data.csv'),index=False)